# Hyperparameter Tuning — CINIC-10 CNN

This notebook runs the hyperparameter sweep and visualizes results.

In [1]:
import os, sys
sys.path.insert(0, os.path.join('..', 'src'))

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from utils import get_device, set_seeds

set_seeds(42)
device = get_device()

DATA_DIR  = os.path.join('..', 'data')
TRAIN_DIR = os.path.join(DATA_DIR, 'train')
VAL_DIR   = os.path.join(DATA_DIR, 'valid')
EPOCHS = 3  # increase for final submission
BATCH_SIZE = 32
print(f'TRAIN_DIR: {TRAIN_DIR}')
print(f'VAL_DIR: {VAL_DIR}')
print(f'Using device: {device}')
print('Setup complete')

TRAIN_DIR: ../data/train
VAL_DIR: ../data/valid
Using device: mps
Setup complete


## 1. Learning Rate Analysis

In [2]:
from model_architecture import create_baseline_cnn
from hyperparameter_analysis import analyze_learning_rates

lr_results = analyze_learning_rates(
    create_baseline_cnn, TRAIN_DIR, VAL_DIR,
    learning_rates=[0.0001, 0.001, 0.01, 0.1],
    epochs=EPOCHS
)
lr_df = pd.DataFrame(lr_results)
print(lr_df[['learning_rate', 'val_accuracy', 'train_accuracy']])

Testing learning rate: 0.0001


Epoch   1/3  loss 1.7741  acc 0.3442  val_loss 1.4828  val_acc 0.4527  *best*


Epoch   2/3  loss 1.5089  acc 0.4440  val_loss 1.3105  val_acc 0.5239  *best*


Epoch   3/3  loss 1.3859  acc 0.4947  val_loss 1.1960  val_acc 0.5678  *best*
Testing learning rate: 0.001


Epoch   1/3  loss 1.6981  acc 0.3657  val_loss 1.3779  val_acc 0.4872  *best*


...

Epoch   2/3  loss 1.7124  acc 0.3495  val_loss 1.5181  val_acc 0.4214  *best*


Epoch   3/3  loss 1.6124  acc 0.3975  val_loss 1.4237  val_acc 0.4623  *best*
Testing learning rate: 0.1


Epoch   1/3  loss 2.3060  acc 0.1340  val_loss 2.1175  val_acc 0.1659  *best*


Epoch   2/3  loss 2.2274  acc 0.1340  val_loss 2.0900  val_acc 0.1798  *best*


Epoch   3/3  loss 2.2177  acc 0.1362  val_loss 2.0781  val_acc 0.1792
   learning_rate  val_accuracy  train_accuracy
0         0.0001      0.567833        0.494733
1         0.0010      0.582656        0.524678
2         0.0100      0.462256        0.397456
3         0.1000      0.179178        0.136178


In [ ]:
_PROJECT_ROOT = os.path.dirname(os.path.dirname(__file__))
DATA_DIR  = os.path.join(_PROJECT_ROOT, "data")
TRAIN_DIR = os.path.join(DATA_DIR, "train")
VAL_DIR   = os.path.join(DATA_DIR, "valid")
TEST_DIR  = os.path.join(DATA_DIR, "test")
RESULTS_DIR = os.path.join(_PROJECT_ROOT, "results")
MODELS_DIR  = os.path.join(_PROJECT_ROOT, "models")

os.makedirs(RESULTS_DIR, exist_ok=True)
pd.DataFrame(lr_results).to_csv(
    os.path.join(RESULTS_DIR, f"hp_learning_rate.csv"), index=False
)

## 2. Batch Size Analysis

In [3]:
from hyperparameter_analysis import analyze_batch_sizes

batch_results = analyze_batch_sizes(
    create_baseline_cnn, TRAIN_DIR, VAL_DIR,
    batch_sizes=[16, 32, 64, 128],
    epochs=EPOCHS
)
batch_df = pd.DataFrame(batch_results)
print(batch_df[['batch_size', 'val_accuracy', 'train_accuracy']])

Testing batch size: 16


Epoch   1/3  loss 1.7631  acc 0.3443  val_loss 1.4315  val_acc 0.4700  *best*


Epoch   2/3  loss 1.4902  acc 0.4554  val_loss 1.2447  val_acc 0.5464  *best*


Epoch   3/3  loss 1.3760  acc 0.5015  val_loss 1.1705  val_acc 0.5751  *best*
Testing batch size: 32


Epoch   1/3  loss 1.7224  acc 0.3585  val_loss 1.4174  val_acc 0.4720  *best*


...

Epoch   3/3  loss 1.3278  acc 0.5166  val_loss 1.1343  val_acc 0.5890  *best*
Testing batch size: 64


Epoch   1/3  loss 1.6712  acc 0.3767  val_loss 1.3890  val_acc 0.4896  *best*


Epoch   2/3  loss 1.3883  acc 0.4895  val_loss 1.2010  val_acc 0.5587  *best*


Epoch   3/3  loss 1.2735  acc 0.5360  val_loss 1.1011  val_acc 0.6008  *best*
Testing batch size: 128


KeyboardInterrupt: 

## 3. Regularization Analysis

In [ ]:
from model_architecture import create_cnn_with_regularization
from hyperparameter_analysis import analyze_regularization_strengths

reg_results = analyze_regularization_strengths(
    create_cnn_with_regularization, TRAIN_DIR, VAL_DIR,
    dropout_rates=[0.1, 0.3, 0.5],
    weight_decays=[1e-4, 1e-3],
    epochs=EPOCHS
)
reg_df = pd.DataFrame(reg_results)
print(reg_df[['dropout_rate', 'weight_decay', 'val_accuracy']])

## 4. Optimizer Comparison

In [ ]:
from hyperparameter_analysis import analyze_optimizers

opt_results = analyze_optimizers(
    create_baseline_cnn, TRAIN_DIR, VAL_DIR,
    optimizers=['adam', 'sgd', 'rmsprop'],
    epochs=EPOCHS
)
opt_df = pd.DataFrame(opt_results)
print(opt_df[['optimizer', 'val_accuracy', 'train_accuracy']])

## 5. Visualizations

In [ ]:
from hyperparameter_analysis import plot_hyperparameter_results

plot_hyperparameter_results(lr_df, title='Learning Rate Analysis')

## Summary

- **Learning rate**: lower LRs (1e-4, 1e-3) typically converge more stably
- **Batch size**: smaller batches provide more gradient updates per epoch
- **Regularization**: higher dropout reduces overfitting
- **Optimizer**: Adam generally outperforms SGD for CNNs